# 08. Decorators and Type Hints

## What you'll learn

- How to annotate functions with **type hints** (`str`, `int`, `Optional`, `Dict`, `Union`, `Any`)
- How to **inspect type hints at runtime** using the `inspect` and `typing` modules
- How **decorators** work under the hood (wrapper functions, `@` syntax, `functools.wraps`)
- How to write **decorators with arguments** (decorator factories)
- How to rebuild a simplified version of the **`@tool` decorator** that agent frameworks use

## Prerequisites

- [01 - Python Fundamentals](01_python_fundamentals.ipynb) — variables, control flow
- [02 - Functions and Scope](02_functions_and_scope.ipynb) — defining functions, closures
- [03 - Data Structures](03_data_structures.ipynb) — dicts, lists
- [04 - Strings and JSON](04_strings_and_json.ipynb) — string formatting, JSON
- [07 - Classes and OOP](07_classes_and_oop.ipynb) — classes, `__init__`, `__repr__`

## Why this matters for agents

Agent frameworks like **smolagents** use decorators and type hints extensively. When you write:

```python
@tool
def web_search(query: str, max_results: int = 5) -> str:
    """Search the web for a query."""
    ...
```

the `@tool` decorator reads the function's **name**, **docstring**, and **type hints** to automatically
generate a tool schema that the LLM can understand. Understanding how this machinery works is
essential for building and debugging agent tools.

> **Further reading:**
> - [Real Python — Primer on Python Decorators](https://realpython.com/primer-on-python-decorators/) — comprehensive, beginner-friendly walkthrough
> - [Python `typing` module docs](https://docs.python.org/3/library/typing.html) — official reference for all type hint constructs
> - [smolagents source code](https://github.com/huggingface/smolagents) — see how `@tool` is implemented in a real agent framework

---
## 1. Type Hints Basics

Type hints (introduced in Python 3.5+) let you annotate what types a function expects and returns.
They don't enforce anything at runtime by default — Python stays dynamically typed — but they serve
as **machine-readable documentation** that tools (and agent frameworks) can inspect.

The basic syntax uses `:` for parameters and `->` for return types.

In [ ]:
from typing import Optional, Dict


def format_message(role: str, content: str) -> Dict[str, str]:
    """Format a chat message as a dict."""
    return {"role": role, "content": content}


def find_tool(name: str, tools: Dict[str, str], default: Optional[str] = None) -> Optional[str]:
    """Look up a tool description by name. Returns None if not found."""
    return tools.get(name, default)


# These work exactly like untyped functions
msg = format_message("user", "What is the weather?")
print(msg)

tool_registry = {"web_search": "Search the web", "calculator": "Do math"}
print(find_tool("web_search", tool_registry))
print(find_tool("unknown_tool", tool_registry))

Notice that `Optional[str]` means "either `str` or `None`". This is very common for agent tool
parameters that have default values.

You can also annotate plain variables, though this is less common in notebooks:

In [ ]:
agent_name: str = "ResearchBot"
max_retries: int = 3
system_prompt: Optional[str] = None

# You can inspect annotations via __annotations__
print(format_message.__annotations__)
print(find_tool.__annotations__)

---
## 2. Advanced Type Hints

Real agent code uses richer type constructs. Here are the ones you'll see most often:

| Hint | Meaning |
|------|---------|
| `List[str]` | A list of strings |
| `Dict[str, Any]` | A dict with string keys and values of any type |
| `Union[str, int]` | Either a string or an int |
| `List[Dict[str, str]]` | A list of dicts (e.g., chat messages) |
| `Any` | No constraint — accept anything |
| `Callable` | A callable (function, method, class) |

In [ ]:
from typing import List, Union, Any, Callable


def build_conversation(messages: List[Dict[str, str]]) -> str:
    """Flatten a list of message dicts into a single prompt string."""
    lines = []
    for msg in messages:
        lines.append(f"{msg['role'].upper()}: {msg['content']}")
    return "\n".join(lines)


def safe_parse(value: Union[str, int]) -> str:
    """Convert a value to string regardless of input type."""
    return str(value)


def run_tool(tool_fn: Callable, args: Dict[str, Any]) -> Any:
    """Call a tool function with keyword arguments."""
    return tool_fn(**args)


# Test with agent-style data
conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Search for Python decorators."},
]
print(build_conversation(conversation))
print()
print(safe_parse(42))
print(safe_parse("hello"))

**Python 3.10+ shorthand:** You can write `str | int` instead of `Union[str, int]`, and
`list[dict[str, str]]` instead of `List[Dict[str, str]]`. We use the `typing` imports here
for compatibility with Python 3.9, but be aware of the newer syntax.

---
## 3. Inspecting Type Hints at Runtime

This is where things get powerful for agent frameworks. Python's `inspect` module and
`typing.get_type_hints()` let you **read** a function's type annotations programmatically.

This is exactly what `@tool` decorators do: they inspect your function to build a schema
the LLM can use to decide which tool to call and with what arguments.

In [ ]:
import inspect
from typing import get_type_hints


def web_search(query: str, max_results: int = 5) -> str:
    """Search the web for a query and return results."""
    return f"Results for '{query}' (top {max_results})"


# Method 1: __annotations__ (simple, but doesn't resolve forward refs)
print("__annotations__:", web_search.__annotations__)

# Method 2: typing.get_type_hints (resolves forward references)
hints = get_type_hints(web_search)
print("get_type_hints:", hints)

# Method 3: inspect.signature (full parameter info including defaults)
sig = inspect.signature(web_search)
print("\nFull signature:", sig)
print("\nParameter details:")
for name, param in sig.parameters.items():
    default = param.default if param.default is not inspect.Parameter.empty else "(required)"
    annotation = param.annotation if param.annotation is not inspect.Parameter.empty else "(none)"
    print(f"  {name}: type={annotation}, default={default}")

Let's build a small utility that extracts a "tool schema" from any typed function — this
is a preview of what we'll do in the capstone section.

In [ ]:
import json


def _type_name(t) -> str:
    """Get a clean name for a type, handling typing generics like List[str]."""
    # For generics like List[str], prefer the origin type name (list, dict)
    origin = getattr(t, '__origin__', None)
    if origin is not None and hasattr(origin, '__name__'):
        return origin.__name__
    # For simple types (int, str) and special forms (Any, Union, Optional)
    return getattr(t, '__name__', None) or str(t)


def extract_tool_info(fn: Callable) -> Dict[str, Any]:
    """Extract name, description, and parameter info from a typed function."""
    sig = inspect.signature(fn)
    hints = get_type_hints(fn)

    params = {}
    for name, param in sig.parameters.items():
        param_info: Dict[str, Any] = {"type": _type_name(hints.get(name, Any))}
        if param.default is not inspect.Parameter.empty:
            param_info["default"] = param.default
            param_info["required"] = False
        else:
            param_info["required"] = True
        params[name] = param_info

    return {
        "name": fn.__name__,
        "description": (fn.__doc__ or "").strip(),
        "parameters": params,
    }


# Test with a simple function
info = extract_tool_info(web_search)
print(json.dumps(info, indent=2))

# Verify it handles typing generics cleanly
from typing import List

def search_with_tags(query: str, tags: List[str] = None, count: int = 5) -> str:
    """Search with optional tag filtering."""
    return f"Searching {query}"

info2 = extract_tool_info(search_with_tags)
print(json.dumps(info2, indent=2))

This is essentially what an LLM sees when deciding which tool to call. The **name** tells it
what the tool does, the **description** gives context, and the **parameters** tell it what
arguments to pass.

---
## 4. Decorators — How They Work

A decorator is just a function that **takes a function as input and returns a new function**.
That's it. The `@` syntax is syntactic sugar.

```python
@my_decorator
def my_func():
    ...

# is exactly the same as:
def my_func():
    ...
my_func = my_decorator(my_func)
```

Let's build a `@log_tool_call` decorator that logs every time an agent tool is invoked.

In [ ]:
import functools


def log_tool_call(fn):
    """Decorator that logs when a tool function is called."""

    @functools.wraps(fn)  # preserves __name__, __doc__, __annotations__
    def wrapper(*args, **kwargs):
        print(f"[TOOL CALL] {fn.__name__}(args={args}, kwargs={kwargs})")
        result = fn(*args, **kwargs)
        print(f"[TOOL RESULT] {result}")
        return result

    return wrapper


@log_tool_call
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))  # simplified for demo


# When we call calculator, it goes through the wrapper
calculator("2 + 3 * 4")

### Why `functools.wraps` matters

Without `@functools.wraps(fn)`, the wrapper would **replace** the original function's metadata.
This is catastrophic for agent frameworks that need to read `__name__` and `__doc__`.

**Agent connection:** Agent frameworks like smolagents read `__name__`, `__doc__`, and `__annotations__` to auto-generate tool schemas. Without `@wraps`, a decorated tool function loses all three — the LLM would see no name, no description, and no parameter types.

In [ ]:
# Because we used @functools.wraps, metadata is preserved:
print(f"Name: {calculator.__name__}")
print(f"Doc:  {calculator.__doc__}")
print(f"Hints: {get_type_hints(calculator)}")


# Without @functools.wraps, we'd lose everything:
def bad_decorator(fn):
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)
    return wrapper  # no @functools.wraps!


@bad_decorator
def broken_tool(query: str) -> str:
    """This docstring will be lost."""
    return query


print(f"\nBad decorator — Name: {broken_tool.__name__}")  # 'wrapper', not 'broken_tool'
print(f"Bad decorator — Doc:  {broken_tool.__doc__}")      # None

---
## 5. Decorators with Arguments

Sometimes you want a decorator that accepts configuration. For example, `@retry(max_attempts=3)`.
This requires an extra level of nesting — a **decorator factory** (a function that returns a decorator).

The pattern is:
```
decorator_factory(args) -> decorator(fn) -> wrapper(*args, **kwargs)
```

In [ ]:
import time


def retry(max_attempts: int = 3, delay: float = 0.1):
    """Decorator factory: retry a function up to max_attempts times."""

    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"  Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_error

        return wrapper

    return decorator


# Simulate a flaky API call that an agent might make
call_count = 0


@retry(max_attempts=3, delay=0.05)
def flaky_api_call(query: str) -> str:
    """Simulate an unreliable API."""
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError("Server unavailable")
    return f"Success: results for '{query}'"


result = flaky_api_call("python decorators")
print(result)

The `@retry` pattern is very common in agent code — LLM APIs are unreliable, and you almost
always want automatic retry logic around external calls.

---
## 6. Putting It Together — Rebuilding `@tool`

Now we have all the pieces. Let's build a `@simple_tool` decorator that does what real agent
frameworks do:

1. **Read** the function's name, docstring, and type hints
2. **Wrap** it in a `SimpleTool` object with `.name`, `.description`, and `.schema` attributes
3. **Keep** it callable so you can still invoke it directly

This is a simplified version of what `smolagents.tool` does under the hood.

In [ ]:
# Map Python types to JSON Schema type names
PYTHON_TYPE_TO_JSON = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}


class SimpleTool:
    """A tool object that wraps a function with metadata for LLM consumption."""

    def __init__(self, fn: Callable):
        self._fn = fn
        self.name = fn.__name__
        self.description = (fn.__doc__ or "").strip()

        # Build schema from type hints
        sig = inspect.signature(fn)
        hints = get_type_hints(fn)

        properties = {}
        required = []

        for param_name, param in sig.parameters.items():
            param_type = hints.get(param_name, Any)
            json_type = PYTHON_TYPE_TO_JSON.get(param_type, "string")
            properties[param_name] = {"type": json_type}

            if param.default is inspect.Parameter.empty:
                required.append(param_name)
            else:
                properties[param_name]["default"] = param.default

        # Remove 'return' from hints — it's not a parameter
        self.schema = {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": properties,
                    "required": required,
                },
            },
        }

    def __call__(self, *args, **kwargs):
        """Make the tool callable — delegates to the wrapped function."""
        return self._fn(*args, **kwargs)

    def __repr__(self):
        params = list(inspect.signature(self._fn).parameters.keys())
        return f"SimpleTool({self.name}({', '.join(params)}))"


def simple_tool(fn: Callable) -> SimpleTool:
    """Decorator that converts a function into a SimpleTool."""
    return SimpleTool(fn)

Now let's apply `@simple_tool` to a few agent-style functions and see the results.

In [ ]:
@simple_tool
def web_search(query: str, max_results: int = 5) -> str:
    """Search the web for information on a given query."""
    return f"Found {max_results} results for: {query}"


@simple_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"Weather in {city}: 22C, sunny"


@simple_tool
def send_email(to: str, subject: str, body: str) -> bool:
    """Send an email to a recipient."""
    print(f"Sending to {to}: {subject}")
    return True


# Inspect the tool objects
tools = [web_search, get_weather, send_email]
for t in tools:
    print(t)
    print(f"  .name        = {t.name}")
    print(f"  .description = {t.description}")
    print()

In [ ]:
# The tools are still callable
print(web_search("Python decorators"))
print(get_weather("San Francisco"))
print(send_email("alice@example.com", "Hello", "How are you?"))

In [ ]:
# The real payoff: auto-generated schemas that an LLM can consume
print("=== Tool Schemas (what the LLM sees) ===")
print()
for t in tools:
    print(json.dumps(t.schema, indent=2))
    print()

This is the core pattern. In a real agent loop, you would:

1. Collect all `SimpleTool` objects and send their `.schema` dicts to the LLM
2. The LLM decides which tool to call and with what arguments
3. You look up the tool by `.name` and call it with the arguments
4. Feed the result back to the LLM

Let's see a quick simulation of this dispatch pattern.

In [ ]:
# Build a tool registry (name -> SimpleTool)
tool_registry = {t.name: t for t in tools}

# Simulate an LLM deciding to call a tool
llm_tool_call = {
    "name": "web_search",
    "arguments": {"query": "best practices for AI agents", "max_results": 3}
}

# Dispatch
tool_name = llm_tool_call["name"]
tool_args = llm_tool_call["arguments"]

if tool_name in tool_registry:
    result = tool_registry[tool_name](**tool_args)
    print(f"Tool '{tool_name}' returned: {result}")
else:
    print(f"Unknown tool: {tool_name}")

---
## Try It Yourself

### Exercise 1: `@timer` decorator

Write a `@timer` decorator that measures and prints how long a function takes to execute.
Use `time.perf_counter()` for accurate timing. Apply it to a function that simulates
an LLM call with `time.sleep(0.1)`.

```python
@timer
def slow_llm_call(prompt: str) -> str:
    time.sleep(0.1)
    return f"Response to: {prompt}"

slow_llm_call("Hello")
# Should print something like: slow_llm_call took 0.1003s
```

In [ ]:
# Exercise 1: Your code here

def timer(fn):
    """Decorator that prints how long fn takes to execute.

    Expected output format: 'fn_name took 0.1003s'
    """
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        # TODO: record start time with time.perf_counter()
        # TODO: call fn(*args, **kwargs) and store result
        # TODO: record end time, print elapsed, return result
        pass
    return wrapper


# @timer
# def slow_llm_call(prompt: str) -> str:
#     """Simulate a slow LLM API call."""
#     time.sleep(0.1)
#     return f"Response to: {prompt}"
#
# result = slow_llm_call("Hello")
# print(result)
# Expected: "slow_llm_call took 0.1003s" then "Response to: Hello"

### Exercise 2: `@validate_types` decorator

Write a `@validate_types` decorator that checks, at call time, whether the provided
arguments match the function's type hints. If a mismatch is found, raise a `TypeError`.

Use `inspect.signature` and `get_type_hints` to read the expected types,
then use `isinstance()` to check actual values.

```python
@validate_types
def create_message(role: str, content: str) -> dict:
    return {"role": role, "content": content}

create_message("user", "hello")       # works
create_message("user", 123)           # raises TypeError
```

*Hint:* Use `sig.bind(*args, **kwargs)` to map positional/keyword args to parameter names.

In [ ]:
# Exercise 2: Your code here

def validate_types(fn):
    """Decorator that checks argument types against type hints at call time.

    Raises TypeError if an argument doesn't match its annotation.

    Approach:
      1. Use inspect.signature(fn) to get the signature
      2. Use get_type_hints(fn) to get the expected types
      3. Inside the wrapper, call sig.bind(*args, **kwargs) to map
         positional/keyword args to parameter names
         (bind() returns a BoundArguments object — use .arguments to get
         a dict of {param_name: value})
      4. Check each value with isinstance(value, expected_type)
    """
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        # TODO: bind args to parameter names using sig.bind(*args, **kwargs)
        # TODO: loop over bound.arguments and check types
        # TODO: raise TypeError on mismatch, then call fn
        pass
    return wrapper


# @validate_types
# def create_message(role: str, content: str) -> dict:
#     return {"role": role, "content": content}
#
# print(create_message("user", "hello"))       # works
# try:
#     create_message("user", 123)              # raises TypeError
# except TypeError as e:
#     print(f"Caught: {e}")

### Exercise 3: Extend `@simple_tool` with full JSON Schema

Our `@simple_tool` generates a basic schema, but it doesn't handle:
- `Optional[str]` (should map to `{"type": ["string", "null"]}`)
- `List[str]` (should map to `{"type": "array", "items": {"type": "string"}}`)
- Parameter descriptions extracted from the docstring (e.g., Google-style or NumPy-style)

Pick **one** of these enhancements and modify the `SimpleTool` class to support it.
Then test it with:

```python
@simple_tool_v2
def search(query: str, tags: List[str] = None, limit: Optional[int] = 10) -> str:
    """Search for documents.

    Args:
        query: The search query string.
        tags: Optional list of tags to filter by.
        limit: Maximum number of results.
    """
    return f"Searching for {query}"
```

In [ ]:
# Exercise 3: Your code here
# Pick ONE enhancement to implement:
#   A) Handle Optional[str] -> {"type": ["string", "null"]}
#   B) Handle List[str]     -> {"type": "array", "items": {"type": "string"}}
#   C) Extract param descriptions from the docstring
#
# Hint for A/B: use typing.get_origin() and typing.get_args()
#   get_origin(Optional[str]) -> Union
#   get_args(Optional[str])   -> (str, NoneType)
#   get_origin(List[str])     -> list
#   get_args(List[str])       -> (str,)

# class SimpleToolV2:
#     """Enhanced SimpleTool that handles generic types."""
#     def __init__(self, fn: Callable):
#         # TODO: copy from SimpleTool and extend the type mapping
#         pass
#
#     def __call__(self, *args, **kwargs):
#         return self._fn(*args, **kwargs)
#
# def simple_tool_v2(fn: Callable) -> SimpleToolV2:
#     return SimpleToolV2(fn)
#
# @simple_tool_v2
# def search(query: str, tags: List[str] = None, limit: Optional[int] = 10) -> str:
#     """Search for documents.
#
#     Args:
#         query: The search query string.
#         tags: Optional list of tags to filter by.
#         limit: Maximum number of results.
#     """
#     return f"Searching for {query}"
#
# print(json.dumps(search.schema, indent=2))

### Exercise 4: Stack multiple decorators

Apply both `@log_tool_call` and `@retry(max_attempts=2)` to the same function.
Experiment with the order — does `@log_tool_call` on top of `@retry` behave differently
from `@retry` on top of `@log_tool_call`? Write both versions and explain what happens.

In [ ]:
# Exercise 4: Your code here
